# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [1]:
import pandas as pd
from IPython.display import display

# ---------------------------------------------------------
# LOAD THE DATASET
# ---------------------------------------------------------

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"

df = pd.read_csv(url)


# ---------------------------------------------------------
# CLEAN COLUMN NAMES
# ---------------------------------------------------------

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

# Remove unnamed columns, if present
df = df.loc[:, ~df.columns.str.startswith("unnamed")]


# ---------------------------------------------------------
# CLEAN NUMERIC COLUMNS
# ---------------------------------------------------------

numeric_columns = [
    "customer_lifetime_value",
    "monthly_premium_auto",
    "total_claim_amount",
    "number_of_policies"
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )


# ---------------------------------------------------------
# CLEAN TEXT COLUMNS
# ---------------------------------------------------------

text_columns = [
    "response",
    "state",
    "gender",
    "education",
    "policy_type",
    "sales_channel"
]

for column in text_columns:
    if column in df.columns:
        df[column] = (
            df[column]
            .astype("string")
            .str.strip()
        )

# Standardize Yes and No values
df["response"] = df["response"].str.title()


# ---------------------------------------------------------
# DISPLAY CLEANED DATA
# ---------------------------------------------------------

print("Cleaned dataset")
print("Shape:", df.shape)

display(df.head())


# =========================================================
# TASK 1
# Create a new DataFrame containing customers who:
# - have total_claim_amount below $1,000
# - responded Yes to the marketing campaign
# =========================================================

low_claim_yes_customers = df[
    (df["total_claim_amount"] < 1000) &
    (df["response"] == "Yes")
].copy()

print("\nTASK 1")
print(
    "Customers with total claims below $1,000 "
    "who responded Yes"
)

print(
    "Number of matching customers:",
    len(low_claim_yes_customers)
)

display(
    low_claim_yes_customers[
        [
            "customer",
            "state",
            "response",
            "policy_type",
            "gender",
            "monthly_premium_auto",
            "customer_lifetime_value",
            "total_claim_amount"
        ]
    ].head(10)
)

print(
    "\nConclusion: "
    f"There are {len(low_claim_yes_customers):,} customers "
    "who responded Yes and had a total claim amount below $1,000."
)


# =========================================================
# TASK 2
# Analyze customers who responded Yes by:
# - policy type
# - gender
# Compare:
# - average monthly premium
# - average customer lifetime value
# - average total claim amount
# =========================================================

yes_customers = df[
    df["response"] == "Yes"
].copy()

profitability_summary = (
    yes_customers
    .groupby(
        ["policy_type", "gender"],
        as_index=False
    )
    .agg(
        customer_count=(
            "customer",
            "count"
        ),
        average_monthly_premium=(
            "monthly_premium_auto",
            "mean"
        ),
        average_customer_lifetime_value=(
            "customer_lifetime_value",
            "mean"
        ),
        average_total_claim_amount=(
            "total_claim_amount",
            "mean"
        )
    )
    .round(2)
)

# Compare customer value with claim amount
profitability_summary["clv_to_claim_ratio"] = (
    profitability_summary[
        "average_customer_lifetime_value"
    ] /
    profitability_summary[
        "average_total_claim_amount"
    ]
).round(2)

profitability_summary = (
    profitability_summary
    .sort_values(
        "clv_to_claim_ratio",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nTASK 2")
print(
    "Average premium, customer lifetime value, "
    "and total claims by policy type and gender"
)

display(profitability_summary)


# Identify important segments
highest_clv_segment = profitability_summary.loc[
    profitability_summary[
        "average_customer_lifetime_value"
    ].idxmax()
]

lowest_claim_segment = profitability_summary.loc[
    profitability_summary[
        "average_total_claim_amount"
    ].idxmin()
]

best_ratio_segment = profitability_summary.loc[
    profitability_summary[
        "clv_to_claim_ratio"
    ].idxmax()
]


print("\nTask 2 conclusions")

print(
    f"The segment with the highest average customer lifetime value is "
    f"{highest_clv_segment['gender']} customers with "
    f"{highest_clv_segment['policy_type']} policies. "
    f"Their average customer lifetime value is "
    f"${highest_clv_segment['average_customer_lifetime_value']:,.2f}."
)

print(
    f"The segment with the lowest average total claim amount is "
    f"{lowest_claim_segment['gender']} customers with "
    f"{lowest_claim_segment['policy_type']} policies. "
    f"Their average claim amount is "
    f"${lowest_claim_segment['average_total_claim_amount']:,.2f}."
)

print(
    f"The strongest segment based on the customer lifetime value "
    f"to claim ratio is "
    f"{best_ratio_segment['gender']} customers with "
    f"{best_ratio_segment['policy_type']} policies. "
    f"Their CLV-to-claim ratio is "
    f"{best_ratio_segment['clv_to_claim_ratio']:.2f}."
)

print(
    "Segments with higher customer lifetime value and lower claims "
    "may appear more profitable and lower risk. However, this is a "
    "simplified comparison because acquisition costs, operating "
    "expenses, and other business costs are not included."
)


# =========================================================
# TASK 3
# Analyze the total number of customers who have policies
# in each state, then filter states with more than 500 customers
# =========================================================

# Count customers by state
state_customer_counts = (
    df.groupby("state")
      .agg(number_of_customers=("customer", "count"))
      .reset_index()
      .sort_values("number_of_customers", ascending=False)
)

print("Total number of customers by state")
display(state_customer_counts)

# Filter states with more than 500 customers
states_above_500 = state_customer_counts[
    state_customer_counts["number_of_customers"] > 500
]

print("\nStates with more than 500 customers")
display(states_above_500)

print(
    f"\nThere are {len(states_above_500)} states with more than "
    "500 customers."
)


# =========================================================
# TASK 4
# Find maximum, minimum, and median customer lifetime
# value by education and gender
# =========================================================

clv_by_education_gender = (
    df.groupby(
        ["education", "gender"],
        as_index=False
    )
    .agg(
        maximum_clv=(
            "customer_lifetime_value",
            "max"
        ),
        minimum_clv=(
            "customer_lifetime_value",
            "min"
        ),
        median_clv=(
            "customer_lifetime_value",
            "median"
        )
    )
    .round(2)
    .sort_values(
        "median_clv",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nTASK 4")
print(
    "Maximum, minimum, and median customer lifetime value "
    "by education and gender"
)

display(clv_by_education_gender)


highest_median_segment = clv_by_education_gender.loc[
    clv_by_education_gender["median_clv"].idxmax()
]

lowest_median_segment = clv_by_education_gender.loc[
    clv_by_education_gender["median_clv"].idxmin()
]

highest_maximum_segment = clv_by_education_gender.loc[
    clv_by_education_gender["maximum_clv"].idxmax()
]


print("\nTask 4 conclusions")

print(
    f"The highest median customer lifetime value belongs to "
    f"{highest_median_segment['gender']} customers with "
    f"{highest_median_segment['education']} education. "
    f"The median CLV is "
    f"${highest_median_segment['median_clv']:,.2f}."
)

print(
    f"The lowest median customer lifetime value belongs to "
    f"{lowest_median_segment['gender']} customers with "
    f"{lowest_median_segment['education']} education. "
    f"The median CLV is "
    f"${lowest_median_segment['median_clv']:,.2f}."
)

print(
    f"The highest maximum customer lifetime value appears among "
    f"{highest_maximum_segment['gender']} customers with "
    f"{highest_maximum_segment['education']} education. "
    f"The maximum CLV is "
    f"${highest_maximum_segment['maximum_clv']:,.2f}."
)

print(
    "The median is more useful for describing a typical customer "
    "because it is less affected by extremely high customer "
    "lifetime values."
)

Cleaned dataset
Shape: (10910, 25)


,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,income,...,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type
0,DK49336,Arizona,4809.216960,No,Basic,College,2/18/11,Employed,M,48029,...,0.0,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,NaN
1,KX64629,California,2228.525238,No,Basic,College,1/18/11,Unemployed,F,0,...,0.0,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,NaN
2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2/10/11,Employed,M,22139,...,0.0,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A
3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,49078,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,1/17/11,Medical Leave,F,23675,...,NaN,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,NaN



TASK 1
Customers with total claims below $1,000 who responded Yes
Number of matching customers: 1399


,customer,state,response,policy_type,gender,monthly_premium_auto,customer_lifetime_value,total_claim_amount
3,XL78013,Oregon,Yes,Corporate Auto,M,97,22332.439460,484.013411
8,FM55990,California,Yes,Personal Auto,M,154,5989.773931,739.200000
15,CW49887,California,Yes,Special Auto,F,114,4626.801093,547.200000
19,NJ54277,California,Yes,Personal Auto,F,94,3746.751625,19.575683
27,MQ68407,Oregon,Yes,Personal Auto,F,111,4376.363592,60.036683
65,LO57874,Oregon,Yes,Corporate Auto,M,63,2300.691547,302.400000
67,KR35099,Washington,Yes,Personal Auto,M,64,7507.455372,231.201886
69,QG27547,Oregon,Yes,Personal Auto,F,78,2867.312197,374.400000
88,CJ51511,Arizona,Yes,Corporate Auto,F,119,13466.920710,571.200000
102,VG56765,Arizona,Yes,Personal Auto,M,61,2471.528431,114.273025



Conclusion: There are 1,399 customers who responded Yes and had a total claim amount below $1,000.

TASK 2
Average premium, customer lifetime value, and total claims by policy type and gender


...


States with more than 500 customers


,state,number_of_customers
1,California,3552
3,Oregon,2909
0,Arizona,1937
2,Nevada,993
4,Washington,888



There are 5 states with more than 500 customers.

TASK 4
Maximum, minimum, and median customer lifetime value by education and gender


,education,gender,maximum_clv,minimum_clv,median_clv
0,High School or Below,M,83325.38,1940.98,6286.73
1,High School or Below,F,55277.45,2144.92,6039.55
2,College,M,61134.68,1918.12,6005.85
3,Master,F,51016.07,2417.78,5729.86
4,Bachelor,F,73225.96,1904.00,5640.51
5,College,F,61850.19,1898.68,5623.61
6,Master,M,50568.26,2272.31,5579.10
7,Doctor,M,32677.34,2267.60,5577.67
8,Bachelor,M,67907.27,1898.01,5548.03
9,Doctor,F,44856.11,2395.57,5332.46



Task 4 conclusions
The highest median customer lifetime value belongs to M customers with High School or Below education. The median CLV is $6,286.73.
The lowest median customer lifetime value belongs to F customers with Doctor education. The median CLV is $5,332.46.
The highest maximum customer lifetime value appears among M customers with High School or Below education. The maximum CLV is $83,325.38.
The median is more useful for describing a typical customer because it is less affected by extremely high customer lifetime values.


In [2]:
# =========================================================
# BONUS TASKS
# =========================================================

# Convert the date column to datetime
df["effective_to_date"] = pd.to_datetime(
    df["effective_to_date"],
    errors="coerce"
)

# Create the month column
df["month"] = df["effective_to_date"].dt.month_name()

# Arrange months in chronological order
month_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December"
]

df["month"] = pd.Categorical(
    df["month"],
    categories=month_order,
    ordered=True
)


# =========================================================
# BONUS 1
# Number of policies sold by state and month
# States as rows and months as columns
# =========================================================

state_month_policy_counts = (
    df.dropna(subset=["state", "month"])
    .groupby(
        ["state", "month"],
        observed=True
    )
    .size()
    .reset_index(name="policies_sold")
)

policies_by_state_month = (
    state_month_policy_counts
    .pivot(
        index="state",
        columns="month",
        values="policies_sold"
    )
    .reindex(columns=month_order)
    .fillna(0)
    .astype(int)
)

# Remove months that contain only zeros
policies_by_state_month = policies_by_state_month.loc[
    :,
    policies_by_state_month.sum(axis=0) > 0
]

print("BONUS 1")
print("Number of policies sold by state and month")

display(policies_by_state_month)


# =========================================================
# BONUS 2
# Policies sold by month for the top 3 states
# =========================================================

# Calculate total policies sold in each state first
state_policy_totals = (
    df.dropna(subset=["state"])
    .groupby(
        "state",
        as_index=False
    )
    .size()
    .rename(
        columns={"size": "total_policies"}
    )
    .sort_values(
        "total_policies",
        ascending=False
    )
    .reset_index(drop=True)
)

# Select the top three states by overall policy count
top_3_states = (
    state_policy_totals
    .head(3)["state"]
    .tolist()
)

# Filter the state-month data to include only the top states
top_3_state_month_data = state_month_policy_counts[
    state_month_policy_counts["state"].isin(top_3_states)
].copy()

# Create the final month-by-state table
top_3_state_month = (
    top_3_state_month_data
    .pivot(
        index="state",
        columns="month",
        values="policies_sold"
    )
    .reindex(
        index=top_3_states,
        columns=month_order
    )
    .fillna(0)
    .astype(int)
)

# Remove months that contain only zeros
top_3_state_month = top_3_state_month.loc[
    :,
    top_3_state_month.sum(axis=0) > 0
]

print("\nBONUS 2")
print("Policies sold by month for the top 3 states")

display(top_3_state_month)

print(
    f"The top three states with the highest total number of "
    f"policies sold are {top_3_states[0]}, "
    f"{top_3_states[1]}, and {top_3_states[2]}."
)

print(
    "The month columns are arranged in chronological order, "
    "and missing state-month combinations are shown as 0."
)


# =========================================================
# BONUS 3
# Customer response rate by marketing channel using melt()
# =========================================================

# Select the columns needed for the analysis
marketing_data = df[
    [
        "customer",
        "response",
        "sales_channel"
    ]
].copy()

# Use melt to convert the channel column to long format
marketing_long = marketing_data.melt(
    id_vars=[
        "customer",
        "response"
    ],
    value_vars=[
        "sales_channel"
    ],
    var_name="channel_type",
    value_name="marketing_channel"
)

# Clean the response values
marketing_long["response"] = (
    marketing_long["response"]
    .fillna("No")
    .astype("string")
    .str.strip()
    .str.title()
)

# Create a Yes-response indicator
marketing_long["responded_yes"] = (
    marketing_long["response"]
    .fillna("No")
    .eq("Yes")
    .astype(int)
)

# Calculate response results for each marketing channel
channel_response_rate = (
    marketing_long
    .dropna(subset=["marketing_channel"])
    .groupby(
        "marketing_channel",
        as_index=False
    )
    .agg(
        total_customers=(
            "customer",
            "count"
        ),
        yes_responses=(
            "responded_yes",
            "sum"
        ),
        response_rate=(
            "responded_yes",
            "mean"
        )
    )
)

# Convert the response rate to a percentage
channel_response_rate["response_rate_percent"] = (
    channel_response_rate["response_rate"] * 100
).round(2)

# Remove the decimal response-rate column
channel_response_rate = (
    channel_response_rate
    .drop(columns="response_rate")
    .sort_values(
        "response_rate_percent",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nBONUS 3")
print("Customer response rate by marketing channel")

display(channel_response_rate)

# Identify the best and weakest channels
best_channel = channel_response_rate.iloc[0]
lowest_channel = channel_response_rate.iloc[-1]

print(
    f"The marketing channel with the highest response rate is "
    f"{best_channel['marketing_channel']}, with a response rate "
    f"of {best_channel['response_rate_percent']:.2f}%."
)

print(
    f"The marketing channel with the lowest response rate is "
    f"{lowest_channel['marketing_channel']}, with a response rate "
    f"of {lowest_channel['response_rate_percent']:.2f}%."
)

print(
    "Channels with higher response rates may be more effective "
    "for future campaigns. However, the company should also "
    "consider campaign cost, customer reach, and customer "
    "lifetime value before allocating its marketing budget."
)

BONUS 1
Number of policies sold by state and month


/var/folders/3_/6lgqyskd5v55bylpxy1jbmzr0000gn/T/ipykernel_73583/1791071802.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["effective_to_date"] = pd.to_datetime(


month,January,February
state,,
Arizona,1008,929
California,1918,1634
Nevada,551,442
Oregon,1565,1344
Washington,463,425



BONUS 2
Policies sold by month for the top 3 states


month,January,February
state,,
California,1918,1634
Oregon,1565,1344
Arizona,1008,929


The top three states with the highest total number of policies sold are California, Oregon, and Arizona.
The month columns are arranged in chronological order, and missing state-month combinations are shown as 0.

BONUS 3
Customer response rate by marketing channel


,marketing_channel,total_customers,yes_responses,response_rate_percent
0,Agent,4121,742,18.01
1,Web,1626,177,10.89
2,Branch,3022,326,10.79
3,Call Center,2141,221,10.32


The marketing channel with the highest response rate is Agent, with a response rate of 18.01%.
The marketing channel with the lowest response rate is Call Center, with a response rate of 10.32%.
Channels with higher response rates may be more effective for future campaigns. However, the company should also consider campaign cost, customer reach, and customer lifetime value before allocating its marketing budget.
